# Step 1 - Data Preparation
**Project:** Customer churn, segmentation, recommendations and retention (Online Retail II)

This notebook turns the raw Excel file into clean, analysis-ready tables that every later step will use.

**What it does**
1. Loads **both** sheets and combines them
2. Standardises text and data types
3. Removes exact duplicates (this also removes the Dec-2010 overlap between the two sheets)
4. Labels every row: real sale / cancellation (return) / non-product code / stock adjustment / bad-debt adjustment / zero-price line
5. Separates the data into clean tables and saves them to the `data/` folder

**Output files**

| File | Contents | Used for |
|---|---|---|
| `clean_sales.csv` | Real purchases with a Customer ID | churn, segmentation, lifetime value, recommenders |
| `clean_returns.csv` | Cancellations with a Customer ID | return-rate feature |
| `sales_no_customer.csv` | Real purchases with no Customer ID | product-level analysis and association rules only |
| `product_lookup.csv` | One clean description per product code | dashboard and recommendations |
| `cleaning_summary.csv` | Row counts for every cleaning step | project report |

Place this notebook next to `online_retail_II.xlsx` and click **Run All**.

## 1. Setup

In [1]:
import os
import numpy as np
import pandas as pd

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_columns = 50

FILE = "online_retail_II.xlsx"   # change the path if the file is somewhere else
OUT = "data"                     # cleaned files are saved here
os.makedirs(OUT, exist_ok=True)

## 2. Load both sheets

In [2]:
sheets = pd.read_excel(FILE, sheet_name=None)      # reads ALL sheets (the file has two)

for name, sh in sheets.items():
    print(f"{name}: {len(sh):,} rows, {sh['InvoiceDate'].min():%d %b %Y} -> {sh['InvoiceDate'].max():%d %b %Y}")

raw = pd.concat(sheets.values(), ignore_index=True)
n_raw = len(raw)
print(f"\nCombined raw rows: {n_raw:,}")

Year 2009-2010: 525,461 rows, 01 Dec 2009 -> 09 Dec 2010
Year 2010-2011: 541,910 rows, 01 Dec 2010 -> 09 Dec 2011

Combined raw rows: 1,067,371


## 3. First look at the raw data

In [3]:
audit = raw.dtypes.astype(str).to_frame("dtype")
audit["missing"] = raw.isna().sum()
audit["missing_%"] = audit["missing"] / len(raw) * 100
display(audit)
display(raw.head())

,dtype,missing,missing_%
Invoice,object,0,0.00
StockCode,object,0,0.00
Description,object,4382,0.41
Quantity,int64,0,0.00
InvoiceDate,datetime64[us],0,0.00
Price,float64,0,0.00
Customer ID,float64,243007,22.77
Country,str,0,0.00


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom


## 4. Standardise text and data types
Stock codes and descriptions are trimmed and upper-cased so that variants such as `85123a` and `85123A` become one product. Customer ID is stored as a whole number that is allowed to be missing.

In [4]:
df = raw.copy()
df.columns = [c.strip() for c in df.columns]

df["Invoice"] = df["Invoice"].astype(str).str.strip().str.upper()
df["StockCode"] = df["StockCode"].astype(str).str.strip().str.upper()
df["Description"] = df["Description"].astype("string").str.strip().str.upper()
df["Customer ID"] = df["Customer ID"].astype("Int64")
df = df.rename(columns={"Customer ID": "CustomerID"})
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

df["Revenue"] = df["Quantity"] * df["Price"]
df.dtypes

Invoice                   str
StockCode                 str
Description            string
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
CustomerID              Int64
Country                   str
Revenue               float64
dtype: object

## 5. Remove exact duplicates
This removes the rows that appear in both sheets (December 2010) and any fully repeated line.

In [5]:
n_dup = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
n_dedup = len(df)
print(f"Duplicate rows removed: {n_dup:,}")
print(f"Rows remaining: {n_dedup:,}")

Duplicate rows removed: 34,335
Rows remaining: 1,033,036


## 6. Label every row
Each row gets exactly one `RowType`. The conditions are checked **in this order** and the first match wins:

| RowType | Rule |
|---|---|
| `bad_debt_adjustment` | Invoice starts with **A** (accounting adjustments, negative prices) |
| `non_product_code` | Stock code is postage, bank charges, fees, test codes, gift vouchers, etc. |
| `cancellation` | Invoice starts with **C** (a return or cancelled order) |
| `stock_adjustment` | Negative quantity on a normal invoice (damaged or lost stock) |
| `zero_or_negative_price` | Price is 0 or below (free items or data errors) |
| `sale` | Everything else: a genuine purchase |

In [6]:
NON_PRODUCT = r"^(?:POST|D|M|DOT|CRUK|BANK CHARGES|PADS|AMAZONFEE|B|S|C2|ADJUST2?|TEST\d+|SP1002|GIFT_.*)$"

is_baddebt = df["Invoice"].str.startswith("A")
is_nonprod = df["StockCode"].str.match(NON_PRODUCT)
is_cancel  = df["Invoice"].str.startswith("C")

conditions = [is_baddebt, is_nonprod, is_cancel, df["Quantity"] < 0, df["Price"] <= 0]
labels = ["bad_debt_adjustment", "non_product_code", "cancellation",
          "stock_adjustment", "zero_or_negative_price"]

df["RowType"] = np.select(conditions, labels, default="sale")

summary = df.groupby("RowType").agg(rows=("Invoice", "size"),
                                    invoices=("Invoice", "nunique"),
                                    value=("Revenue", "sum"))
summary["rows_%"] = summary["rows"] / len(df) * 100
summary.sort_values("rows", ascending=False)

,rows,invoices,value,rows_%
RowType,,,,
sale,1003338,39516,"19,643,846.87",97.13
cancellation,17914,7405,"-716,425.97",1.73
non_product_code,5815,5481,"75,726.88",0.56
stock_adjustment,3392,3392,0.00,0.33
zero_or_negative_price,2571,1938,0.00,0.25
bad_debt_adjustment,6,6,"-147,614.08",0.00


### Sanity check: which codes were treated as non-products, and what is left that is not a 5-digit code?
Real products have 5-digit stock codes (sometimes with a letter on the end). Anything else is shown here so you can confirm it deserves to stay.

In [7]:
print("Non-product codes removed:")
display(df[df["RowType"] == "non_product_code"]
        .groupby("StockCode").agg(rows=("Invoice", "size"), example=("Description", "first"))
        .sort_values("rows", ascending=False))

odd = df[(df["RowType"] == "sale") & ~df["StockCode"].str.match(r"^\d{5}")]
print("Kept as sales although the code is not 5 digits (review):")
display(odd.groupby("StockCode").agg(rows=("Invoice", "size"), example=("Description", "first"))
        .sort_values("rows", ascending=False).head(15))

Non-product codes removed:


,rows,example
StockCode,,
POST,2086,POSTAGE
DOT,1425,DOTCOM POSTAGE
M,1392,MANUAL
C2,277,CARRIAGE
D,173,DISCOUNT
S,101,SAMPLES
BANK CHARGES,100,BANK CHARGES
ADJUST,67,ADJUSTMENT BY JOHN ON 26/01/2010 16
AMAZONFEE,36,AMAZON FEE


Kept as sales although the code is not 5 digits (review):


,rows,example
StockCode,,
DCGS0058,30,MISO PRETTY GUM
DCGSSGIRL,23,GIRLS PARTY BAG
DCGSSBOY,21,BOYS PARTY BAG
DCGS0076,14,SUNJAR LED NIGHT NIGHT LIGHT
DCGS0003,13,BOXED GLASS ASHTRAY
DCGS0069,5,OOH LA LA DOGS COLLAR
DCGS0004,4,HAYNES CAMPER SHOULDER BAG
DCGS0072,3,CAT CAMOUFLAGUE COLLAR
DCGS0066N,2,NAVY CUDDLES DOG HOODIE


## 7. Split into the clean tables
- **sales**: real purchases with a Customer ID (the main table for the project)
- **sales_no_customer**: real purchases without a Customer ID. They cannot be used for customer-level work but still show which products are bought together
- **returns**: cancellations with a Customer ID (they become a return-rate feature)

In [8]:
sales_all = df[df["RowType"] == "sale"]

sales = sales_all[sales_all["CustomerID"].notna()].copy()
sales_no_customer = sales_all[sales_all["CustomerID"].isna()].drop(columns="CustomerID").copy()

returns_all = df[df["RowType"] == "cancellation"]
returns = returns_all[returns_all["CustomerID"].notna()].copy()

sales["CustomerID"] = sales["CustomerID"].astype("int64")
returns["CustomerID"] = returns["CustomerID"].astype("int64")

# helpful extra columns
returns["ReturnQty"] = -returns["Quantity"]
returns["ReturnValue"] = -returns["Revenue"]
for t in (sales, sales_no_customer, returns):
    t["InvoiceMonth"] = t["InvoiceDate"].dt.to_period("M").astype(str)

print(f"sales             : {len(sales):,} rows, {sales['CustomerID'].nunique():,} customers")
print(f"sales_no_customer : {len(sales_no_customer):,} rows")
print(f"returns           : {len(returns):,} rows, {returns['CustomerID'].nunique():,} customers")

sales             : 776,577 rows, 5,852 customers
sales_no_customer : 226,761 rows
returns           : 17,586 rows, 2,444 customers


## 8. One clean description per product
Some stock codes have several spellings of the description. For the dashboard and recommender we use the most common one.

In [9]:
product_lookup = (sales_all.dropna(subset=["Description"])
                  .groupby(["StockCode", "Description"]).size().reset_index(name="n")
                  .sort_values(["StockCode", "n"], ascending=[True, False])
                  .drop_duplicates("StockCode")[["StockCode", "Description"]]
                  .reset_index(drop=True))

desc_map = dict(zip(product_lookup["StockCode"], product_lookup["Description"]))
for t in (sales, sales_no_customer, returns):
    t["Description"] = t["StockCode"].map(desc_map).fillna(t["Description"])

# unifying descriptions can make two lines identical, so drop any new duplicates
n_dup2 = int(sales.duplicated().sum() + sales_no_customer.duplicated().sum() + returns.duplicated().sum())
sales = sales.drop_duplicates().reset_index(drop=True)
sales_no_customer = sales_no_customer.drop_duplicates().reset_index(drop=True)
returns = returns.drop_duplicates().reset_index(drop=True)
print(f"Extra duplicates removed after description clean-up: {n_dup2}")

print(f"Products in lookup: {len(product_lookup):,}")
product_lookup.head()

Extra duplicates removed after description clean-up: 2
Products in lookup: 4,723


,StockCode,Description
0,10002,INFLATABLE POLITICAL GLOBE
1,10002R,ROBOT PENCIL SHARPNER
2,10080,GROOVY CACTUS INFLATABLE
3,10109,BENDY COLOUR PENCILS
4,10120,DOGGY RUBBER


## 9. Validation checks
Every line should say `True`. If one says `False`, fix it before moving on.

In [10]:
checks = {
    "sales: no missing CustomerID":      sales["CustomerID"].notna().all(),
    "sales: all quantities > 0":         (sales["Quantity"] > 0).all(),
    "sales: all prices > 0":             (sales["Price"] > 0).all(),
    "sales: no duplicate rows":          not sales.duplicated().any(),
    "sales: no missing description":     sales["Description"].notna().all(),
    "returns: all quantities < 0":       (returns["Quantity"] < 0).all(),
    "returns: no missing CustomerID":    returns["CustomerID"].notna().all(),
    "row counts add up":                 len(df) == df["RowType"].value_counts().sum(),
}
for k, v in checks.items():
    print(f"{str(bool(v)):5}  {k}")

share = returns["CustomerID"].isin(sales["CustomerID"]).mean()
print(f"\nShare of returned lines whose customer also has purchases: {share:.1%}")

True   sales: no missing CustomerID
True   sales: all quantities > 0
True   sales: all prices > 0
True   sales: no duplicate rows
True   sales: no missing description
True   returns: all quantities < 0
True   returns: no missing CustomerID
True   row counts add up

Share of returned lines whose customer also has purchases: 99.5%


## 10. Cleaning summary (for your report)

In [11]:
cleaning_summary = pd.DataFrame([
    ("Raw rows (both sheets)",                              n_raw),
    ("Exact duplicates removed",                            -n_dup),
    ("Rows after de-duplication",                           n_dedup),
    ("Removed: bad-debt adjustments",                       -int((df["RowType"] == "bad_debt_adjustment").sum())),
    ("Removed: non-product codes",                          -int((df["RowType"] == "non_product_code").sum())),
    ("Removed: stock adjustments",                          -int((df["RowType"] == "stock_adjustment").sum())),
    ("Removed: zero or negative price",                     -int((df["RowType"] == "zero_or_negative_price").sum())),
    ("Cancellations (kept separately, with Customer ID)",   len(returns)),
    ("Cancellations dropped (no Customer ID)",              len(returns_all) - len(returns)),
    ("Valid sales without Customer ID (kept separately)",   len(sales_no_customer)),
    ("Duplicates found after description clean-up",         -n_dup2),
    ("CLEAN SALES (with Customer ID)",                      len(sales)),
], columns=["step", "rows"])

cleaning_summary["rows"] = cleaning_summary["rows"].map("{:,}".format)
display(cleaning_summary)

print(f"\nDate range of clean sales: {sales['InvoiceDate'].min():%d %b %Y} -> {sales['InvoiceDate'].max():%d %b %Y}")
print(f"Customers: {sales['CustomerID'].nunique():,}   Products: {sales['StockCode'].nunique():,}   Invoices: {sales['Invoice'].nunique():,}")

,step,rows
0,Raw rows (both sheets),"1,067,371"
1,Exact duplicates removed,"-34,335"
2,Rows after de-duplication,"1,033,036"
3,Removed: bad-debt adjustments,-6
4,Removed: non-product codes,"-5,815"
5,Removed: stock adjustments,"-3,392"
6,Removed: zero or negative price,"-2,571"
7,"Cancellations (kept separately, with Customer ID)","17,586"
8,Cancellations dropped (no Customer ID),328
9,Valid sales without Customer ID (kept separately),"226,761"



Date range of clean sales: 01 Dec 2009 -> 09 Dec 2011
Customers: 5,852   Products: 4,619   Invoices: 36,594


## 11. Save the clean files

In [12]:
sales.to_csv(f"{OUT}/clean_sales.csv", index=False)
returns.to_csv(f"{OUT}/clean_returns.csv", index=False)
sales_no_customer.to_csv(f"{OUT}/sales_no_customer.csv", index=False)
product_lookup.to_csv(f"{OUT}/product_lookup.csv", index=False)
cleaning_summary.to_csv(f"{OUT}/cleaning_summary.csv", index=False)

for f in sorted(os.listdir(OUT)):
    print(f"{f:25s} {os.path.getsize(os.path.join(OUT, f)) / 1e6:6.1f} MB")

sales.head()

clean_returns.csv            2.3 MB
clean_sales.csv             85.4 MB
cleaning_summary.csv         0.0 MB
product_lookup.csv           0.2 MB
sales_no_customer.csv       23.3 MB


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,Revenue,RowType,InvoiceMonth
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40,sale,2009-12
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,sale,2009-12
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,sale,2009-12
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80,sale,2009-12
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00,sale,2009-12


## Next step
Step 2 is **exploratory analysis** (Objective 1) and then **churn label and feature engineering** (Objective 2). Both will start by loading `data/clean_sales.csv` and `data/clean_returns.csv`:

```python
sales = pd.read_csv("data/clean_sales.csv", parse_dates=["InvoiceDate"])
returns = pd.read_csv("data/clean_returns.csv", parse_dates=["InvoiceDate"])
```